# Tennis Match Winning Prediction based on Bradley-Terry Model (BT)


This notebook is an end-to-end quantitative modeling pipeline for predicting WTA and ATP tennis match outcomes using the Bradley-Terry model.

### Key Technical Implementations
1. **Mathematical Probability Mapping**: Instead of predicting match outcomes directly, the model predicts match outcome based on *game-winning probability*. In this way, we make use of the richer information from historical game scores. Custom combinatorics are used to mathematically scale game-winning probabilities into *match-winning probabilities* based on the exact structure of best-of-3 or best-of-5 sets.
2. **Vectorized Optimization**: Instead of using generic minimization algorithms directly on rows, both models implement gradient descent for optimisation. This drastically cuts down optimization time.
3. **Priors**: The Bayesian version uses a normal prior scaled around average pre-existing player rankings to provide a robust regularization anchor.
4. **Strict Walk-Forward Evaluation**: To mimic a realistic betting/trading simulation, the models evaluate the latest half-season (2025 H2) tournament by tournament. Before each tournament starts, the model is refitted strictly using matches completed prior to the start date. 
5. **Bookmaker Benchmarking**: Final accuracy and log-loss are benchmarked directly against the implied probabilities generated by Bet365 odds.

---

### 1. Configuration & Mathematical Probability Helpers
Game-win probabilities are converted to Set-win probabilities, and Set-win probabilities to Match-win probabilities based on the rules of Tennis sets. 

In [106]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO
from math import comb
from scipy.special import expit

BASE_URLS = ['https://www.tennis-data.co.uk', 'http://www.tennis-data.co.uk']
SPORT = 'WTA' #use "ATP" for male tennis
YEARS = [2022, 2023, 2024, 2025]
HALF_CUTOFF = pd.Timestamp('2025-07-01')


def set_score_distribution_from_game_prob(p_game, p_tiebreak=None):
    if not (0 <= p_game <= 1):
        raise ValueError('p_game must be in [0, 1]')

    p_tb = p_game if p_tiebreak is None else p_tiebreak
    if not (0 <= p_tb <= 1):
        raise ValueError('p_tiebreak must be in [0, 1]')

    p = p_game
    q = 1 - p

    dist = {}
    for k in range(5):
        dist[(6, k)] = comb(5 + k, k) * (p**6) * (q**k)
        dist[(k, 6)] = comb(5 + k, k) * (q**6) * (p**k)

    dist[(7, 5)] = comb(10, 5) * (p**7) * (q**5)
    dist[(5, 7)] = comb(10, 5) * (q**7) * (p**5)

    prob_reach_6_6 = comb(10, 5) * 2 * (p**6) * (q**6)
    dist[(7, 6)] = prob_reach_6_6 * p_tb
    dist[(6, 7)] = prob_reach_6_6 * (1 - p_tb)
    return dist


def set_win_prob_from_game_prob(p_game, p_tiebreak=None):
    dist = set_score_distribution_from_game_prob(p_game, p_tiebreak=p_tiebreak)
    return sum(prob for (w, l), prob in dist.items() if w > l)


def match_win_prob_from_game_prob(p_game, best_of=3, p_tiebreak=None):
    p_set = set_win_prob_from_game_prob(p_game, p_tiebreak=p_tiebreak)
    q_set = 1 - p_set

    try:
        best_of_int = int(best_of)
    except (TypeError, ValueError):
        best_of_int = 3

    if best_of_int == 5:
        return p_set**3 + 3 * (p_set**3) * q_set + 6 * (p_set**3) * (q_set**2)

    return p_set**2 + 2 * (p_set**2) * q_set

---
### 2. Data Fetching & Preprocessing
Downloads raw `.xlsx` files from `tennis-data.co.uk`, cleans records, filters to completed matches, and applies a strict Train/Test boundary.

In [107]:
def download_tennis_data(years, sport='WTA'):
    all_data = []
    sport_code = 'w' if sport.upper() == 'WTA' else ''
    headers = {'User-Agent': 'Mozilla/5.0'}

    for year in years:
        candidate_urls = [f'{base}/{year}{sport_code}/{year}.xlsx' for base in BASE_URLS]
        downloaded = False

        for url in candidate_urls:
            try:
                response = requests.get(url, timeout=15, headers=headers)
                response.raise_for_status()
                df = pd.read_excel(BytesIO(response.content))
                df['Year'] = year
                df['Sport'] = sport.upper()
                all_data.append(df)
                downloaded = True
                print(f'✓ {year} loaded')
                break
            except Exception:
                continue

        if not downloaded:
            print(f'✗ {year} unavailable')

    if not all_data:
        raise ValueError('No data downloaded.')

    return pd.concat(all_data, ignore_index=True)

In [108]:
raw = download_tennis_data(YEARS, sport=SPORT)

data = (raw
    .copy()
    .assign(Date=lambda d: pd.to_datetime(d['Date'], errors='coerce'))
    .dropna(subset=['Date', 'Winner', 'Loser'])
)

if 'Comment' in data.columns:
    data = data[data['Comment'].astype(str).str.lower().eq('completed')].copy()

if 'Surface' in data.columns:
    data['Surface'] = data['Surface'].fillna('Hard')

train_df = data[((data['Year'] < 2025) | ((data['Year'] == 2025) & (data['Date'] < HALF_CUTOFF)))].copy()
test_df = data[(data['Year'] == 2025) & (data['Date'] >= HALF_CUTOFF)].copy()

print('Train size:', len(train_df))
print('Test size:', len(test_df))

✓ 2022 loaded
✓ 2023 loaded
✓ 2024 loaded
✓ 2025 loaded
Train size: 8442
Test size: 1006


---
### 3. Model Definition: Base Bradley-Terry (MLE)
Standard Maximum Likelihood Estimation on player strength `theta`. Includes optimized matrix formulations for computing log-likelihood and exact analytic gradients.

In [109]:
from scipy.optimize import minimize

class BaseBradleyTerry:
    def __init__(self, min_matches=5):
        self.min_matches = min_matches
        self.players = None
        self.player_to_idx = None
        self.theta = None

    def _prepare_players(self, df):
        counts = pd.concat([df['Winner'], df['Loser']]).value_counts()
        self.players = counts[counts >= self.min_matches].index.tolist()
        self.player_to_idx = {p: i for i, p in enumerate(self.players)}

    def fit(self, df):
        self._prepare_players(df)
        work = df[df['Winner'].isin(self.player_to_idx) & df['Loser'].isin(self.player_to_idx)].copy()

        if len(work) == 0:
            self.theta = np.zeros(len(self.players))
            return self

        winner_idx = work['Winner'].map(self.player_to_idx).to_numpy(dtype=int)
        loser_idx = work['Loser'].map(self.player_to_idx).to_numpy(dtype=int)

        w_games = np.zeros(len(work), dtype=float)
        l_games = np.zeros(len(work), dtype=float)
        for i in range(1, 6):
            w_col = f'W{i}'
            l_col = f'L{i}'
            if w_col in work.columns:
                w_games += pd.to_numeric(work[w_col], errors='coerce').fillna(0).to_numpy(dtype=float)
            if l_col in work.columns:
                l_games += pd.to_numeric(work[l_col], errors='coerce').fillna(0).to_numpy(dtype=float)

        total_games = w_games + l_games
        no_game_rows = total_games <= 0
        if np.any(no_game_rows):
            w_games[no_game_rows] = 1.0
            l_games[no_game_rows] = 0.0
            total_games[no_game_rows] = 1.0

        n_players = len(self.players)

        def neg_log_likelihood(params):
            theta = np.concatenate([[0], params])
            diff = theta[winner_idx] - theta[loser_idx]
            p = expit(diff)
            return -np.sum(w_games * np.log(np.maximum(p, 1e-10)) + l_games * np.log(np.maximum(1 - p, 1e-10)))

        def gradient(params):
            theta = np.concatenate([[0], params])
            diff = theta[winner_idx] - theta[loser_idx]
            p = expit(diff)
            grad_contrib = total_games * p - w_games
            grad = np.zeros(n_players)
            np.add.at(grad, winner_idx, grad_contrib)
            np.add.at(grad, loser_idx, -grad_contrib)
            return grad[1:]

        init_params = np.zeros(n_players - 1)
        result = minimize(
            neg_log_likelihood,
            init_params,
            method='L-BFGS-B',
            jac=gradient,
            options={'gtol': 1e-5}
        )

        self.theta = np.concatenate([[0], result.x])
        self.theta -= self.theta.mean() # center at 0
        return self

    def predict_proba(self, winner, loser, best_of=3):
        iw = self.player_to_idx.get(winner)
        il = self.player_to_idx.get(loser)
        if iw is None or il is None:
            return np.nan
        p_game = float(expit(self.theta[iw] - self.theta[il]))
        return float(match_win_prob_from_game_prob(p_game, best_of=best_of))

---
### 4. Model Definition: Bayesian Bradley-Terry (MAP)
Extends the baseline by extracting `WRank` and `LRank` (Player Ranks) during training to build normally distributed regularization priors. Updates the objective function and gradient to incorporate the prior term penalty.

In [110]:
class BayesianBradleyTerry(BaseBradleyTerry):
    def __init__(self, min_matches=5, prior_sigma=1.0):
        super().__init__(min_matches=min_matches)
        self.prior_sigma = prior_sigma
        self.prior_mean = None

    def fit(self, df):
        self._prepare_players(df)
        work = df[df['Winner'].isin(self.player_to_idx) & df['Loser'].isin(self.player_to_idx)].copy()

        if len(work) == 0:
            self.prior_mean = np.zeros(len(self.players))
            self.theta = self.prior_mean.copy()
            return self

        winner_idx = work['Winner'].map(self.player_to_idx).to_numpy(dtype=int)
        loser_idx = work['Loser'].map(self.player_to_idx).to_numpy(dtype=int)

        w_games = np.zeros(len(work), dtype=float)
        l_games = np.zeros(len(work), dtype=float)
        for i in range(1, 6):
            w_col = f'W{i}'
            l_col = f'L{i}'
            if w_col in work.columns:
                w_games += pd.to_numeric(work[w_col], errors='coerce').fillna(0).to_numpy(dtype=float)
            if l_col in work.columns:
                l_games += pd.to_numeric(work[l_col], errors='coerce').fillna(0).to_numpy(dtype=float)

        total_games = w_games + l_games
        no_game_rows = total_games <= 0
        if np.any(no_game_rows):
            w_games[no_game_rows] = 1.0
            l_games[no_game_rows] = 0.0
            total_games[no_game_rows] = 1.0

        rank_map = {}
        if 'WRank' in work.columns and 'LRank' in work.columns:
            tmp = pd.concat([
                work[['Winner', 'WRank']].rename(columns={'Winner': 'Player', 'WRank': 'Rank'}),
                work[['Loser', 'LRank']].rename(columns={'Loser': 'Player', 'LRank': 'Rank'})
            ], ignore_index=True)
            tmp['Rank'] = pd.to_numeric(tmp['Rank'], errors='coerce')
            rank_map = tmp.groupby('Player', as_index=True)['Rank'].mean().to_dict()

        prior = np.zeros(len(self.players), dtype=float)
        for p, i in self.player_to_idx.items():
            r = rank_map.get(p, 100) # fillna with 100 like old code
            if pd.isna(r):
                r = 100
            prior[i] = -np.log(r + 1)
            
        prior -= prior[0] # align with old code
        self.prior_mean = prior

        n_players = len(self.players)
        reg_scale = max(self.prior_sigma ** 2, 1e-8)

        def neg_log_posterior(params):
            theta = np.concatenate([[0], params])
            diff = theta[winner_idx] - theta[loser_idx]
            p = expit(diff)
            nll = -np.sum(w_games * np.log(np.maximum(p, 1e-10)) + l_games * np.log(np.maximum(1 - p, 1e-10)))
            
            # Use original formulation for prior
            prior_mu_optim = self.prior_mean[1:]
            prior_term = (1.0 / (2 * reg_scale)) * np.sum((params - prior_mu_optim) ** 2)
            
            return nll + prior_term

        def gradient(params):
            theta = np.concatenate([[0], params])
            diff = theta[winner_idx] - theta[loser_idx]
            p = expit(diff)
            grad_contrib = total_games * p - w_games
            
            grad_nll = np.zeros(n_players)
            np.add.at(grad_nll, winner_idx, grad_contrib)
            np.add.at(grad_nll, loser_idx, -grad_contrib)
            
            prior_mu_optim = self.prior_mean[1:]
            grad_prior = (2.0 / (2 * reg_scale)) * (params - prior_mu_optim)
            
            return grad_nll[1:] + grad_prior

        init_params = self.prior_mean[1:].copy()
        result = minimize(
            neg_log_posterior,
            init_params,
            method='L-BFGS-B',
            jac=gradient,
            options={'gtol': 1e-6}
        )

        self.theta = np.concatenate([[0], result.x])
        return self

---
### 5. Strict Walk-Forward Evaluation
Iterates over each valid tournament in the test set temporally. For each tournament:
1. Filters the training set to only contain matches resolving *before* the current tournament's start date (prevents data leakage).
2. Fits both models.

In [111]:
def evaluate_predictions(df, prob_col):
    valid = df[prob_col].notna()
    p = df.loc[valid, prob_col].clip(1e-6, 1 - 1e-6)
    y = np.ones(len(p))  # row is winner vs loser
    acc = float((p > 0.5).mean()) if len(p) else np.nan
    logloss = float(-(np.log(p)).mean()) if len(p) else np.nan
    brier = float(((p - y) ** 2).mean()) if len(p) else np.nan
    return {'Accuracy': acc, 'LogLoss': logloss, 'Brier': brier, 'Coverage': len(p) / len(df)}

# Walk-forward by tournament in 2025 H2:
# each tournament is fit using all matches before its start date.
if 'Tournament' in test_df.columns:
    tournament_calendar = (
        test_df.dropna(subset=['Tournament'])
        .groupby('Tournament', as_index=False)['Date']
        .min()
        .rename(columns={'Date': 'TournamentStart'})
        .sort_values(['TournamentStart', 'Tournament'])
        .reset_index(drop=True)
    )
else:
    tournament_calendar = pd.DataFrame([
        {'Tournament': 'ALL_H2', 'TournamentStart': test_df['Date'].min()}
    ])

scored_parts = []
for _, row in tournament_calendar.iterrows():
    t_name = row['Tournament']
    t_start = pd.Timestamp(row['TournamentStart'])

    train_local = data[data['Date'] < t_start].copy()
    if 'Tournament' in test_df.columns:
        test_local = test_df[test_df['Tournament'] == t_name].copy()
    else:
        test_local = test_df.copy()

    if len(train_local) == 0 or len(test_local) == 0:
        continue

    baseline_local = BaseBradleyTerry(min_matches=5).fit(train_local)
    bayesian_local = BayesianBradleyTerry(min_matches=5, prior_sigma=1.0).fit(train_local)

    part = test_local.copy()
    best_of_values = pd.to_numeric(part['Best of'], errors='coerce').fillna(3).astype(int) if 'Best of' in part.columns else pd.Series([3] * len(part), index=part.index)

    part['pred_Baseline'] = [
        baseline_local.predict_proba(w, l, best_of=b)
        for w, l, b in zip(part['Winner'], part['Loser'], best_of_values)
    ]
    part['pred_Bayesian'] = [
        bayesian_local.predict_proba(w, l, best_of=b)
        for w, l, b in zip(part['Winner'], part['Loser'], best_of_values)
    ]
    part['TournamentStart'] = t_start
    scored_parts.append(part)

if not scored_parts:
    raise ValueError('No walk-forward tournament segments were produced.')

scored = pd.concat(scored_parts, ignore_index=True)

summary_rows = []
for col, name in [('pred_Baseline', 'Baseline BT'), ('pred_Bayesian', 'Bayesian BT')]:
    m = evaluate_predictions(scored, col)
    summary_rows.append({'Model': name, **m})

model_summary = pd.DataFrame(summary_rows).sort_values('LogLoss').reset_index(drop=True)
model_summary

,Model,Accuracy,LogLoss,Brier,Coverage
0,Baseline BT,0.658986,0.634386,0.219927,0.862823
1,Bayesian BT,0.664747,0.662606,0.221937,0.862823


---
### 6. Results: Bookmaker Benchmark
Compares model outputs gainst implied winner probability from Bet365.

In [112]:
# Bookmaker benchmark (B365 implied probability)
if 'B365W' in scored.columns and 'B365L' in scored.columns:
    w = pd.to_numeric(scored['B365W'], errors='coerce')
    l = pd.to_numeric(scored['B365L'], errors='coerce')
    valid = w.notna() & l.notna() & (w > 0) & (l > 0)
    inv_w = 1 / w[valid]
    inv_l = 1 / l[valid]
    scored.loc[valid, 'pred_B365'] = inv_w / (inv_w + inv_l)

    b365_metrics = evaluate_predictions(scored, 'pred_B365')
    benchmark_row = pd.DataFrame([{'Model': 'B365 implied probability', **b365_metrics}])
    compare_with_b365 = pd.concat([model_summary, benchmark_row], ignore_index=True)
    compare_with_b365 = compare_with_b365.sort_values('Accuracy', ascending=False).reset_index(drop=True)
    display(compare_with_b365)
else:
    print('B365 columns not found in test set.')

,Model,Accuracy,LogLoss,Brier,Coverage
0,B365 implied probability,0.677031,0.591203,0.202624,0.991054
1,Bayesian BT,0.664747,0.662606,0.221937,0.862823
2,Baseline BT,0.658986,0.634386,0.219927,0.862823


---
### 7. Results: Accuracy Breakdowns by Surface
Evaluates the model's un-tuned implicit performance characteristics across the varying domains of clay, grass, and hard courts.

In [95]:
# Accuracy by surface
rows = []
for col, name in [('pred_Baseline', 'Baseline BT'), ('pred_Bayesian', 'Bayesian BT'), ('pred_B365', 'B365 implied probability')]:
    if col not in scored.columns:
        continue
    tmp = scored[scored[col].notna()].copy()
    tmp['Correct'] = tmp[col] > 0.5
    for surface, g in tmp.groupby('Surface', dropna=False):
        rows.append({'Model': name, 'Surface': surface, 'Matches': len(g), 'Accuracy': float(g['Correct'].mean())})

surface_accuracy = pd.DataFrame(rows)
surface_accuracy_pivot = surface_accuracy.pivot(index='Surface', columns='Model', values='Accuracy')
surface_accuracy_pivot

Model,B365 implied probability,Baseline BT,Bayesian BT
Surface,,,
Clay,0.628571,0.566265,0.590361
Grass,0.747368,0.731707,0.719512
Hard,0.654734,0.629630,0.630864
